# Pratica 02 — Testo grezzo e tokenizzazione

**Corso:** NLP Classico con Python · **Lezione 02** · **Tempo stimato:** 60 minuti
**Notebook di lezione:** `notebooks_clase/clase02_texto_crudo_tokenizacion_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- I testi e i risultati attesi sono nella cella di dati che segue il setup. Sono brevi e scritti per il corso (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Non serve nessuna libreria oltre a Python (usiamo `re`, `unicodedata` e `collections`).

---
## Cella 0 — Setup e dati

In [ ]:
import re
import unicodedata
from collections import Counter

print("Librerie caricate correttamente.")

In [ ]:
# datos-es
PALABRA_NFC = "canción"
PALABRA_NFD = "cancio" + "́" + "n"

CASOS_LIMPIEZA = [
    ("¡Qué   bueno!!!  https://t.co/x1", "¡Qué bueno!"),
    ("@tienda Llegó tardeeee...", "Llegó tarde..."),
    ("Cuesta 1000 euros", "Cuesta 1000 euros"),
    ("Non è  vero ,  davvero???", "Non è vero, davvero?"),
    ("Fantastico #consigliato www.sito.it", "Fantastico"),
]

CASOS_TOKEN = [
    ("¿Dónde está l'uomo?", ["¿", "Dónde", "está", "l'", "uomo", "?"]),
    ("Costa 4,99 euro.", ["Costa", "4,99", "euro", "."]),
    ("Ho un'ora, dell'amico.", ["Ho", "un'", "ora", ",", "dell'", "amico", "."]),
    ("Vamos al mercado, ¡ya!", ["Vamos", "al", "mercado", ",", "¡", "ya", "!"]),
    ("Il 12.5 costa 1.250,50 euro", ["Il", "12.5", "costa", "1.250,50", "euro"]),
]

TOKENS_ES = "el sol sale y el sol se pone y el día pasa".split()
TOKENS_IT = "il gatto dorme e il cane dorme ma il topo no".split()

PARES = [
    ("BUEN PRODUCTO, LLEGÓ RÁPIDO", "Buen producto, llegó rápido"),
    ("Muy bueno!!  https://t.co/a1", "Muy bueno!"),
    ("Nos gustó mucho 😍", "Nos gustó mucho"),
    ("El envio llego tarde", "El envío llegó tarde"),
    ("Es un desatre total", "Es un desastre total"),
    ("Todo perfecto", "Todo perfecto"),
    ("Me encantó   el diseño", "Me encantó el diseño"),
    ("Precio justo, pero la pantalla se raya facil", "Precio justo, pero la pantalla se raya fácil"),
    ("Llegó a tiempo. Gracias @soporte", "Llegó a tiempo. Gracias"),
    ("Recomendable 100%", "Recomendable 100%"),
    ("OTTIMO SERVIZIO", "Ottimo servizio"),
    ("Consegna veloce   e imballo perfetto", "Consegna veloce e imballo perfetto"),
    ("Non e' male, ma nn lo ricomprerei", "Non è male, ma non lo ricomprerei"),
    ("Pacco arrivato rotto 😡 https://goo.gl/x", "Pacco arrivato rotto"),
    ("Bellissimooo, lo consiglio", "Bellissimo, lo consiglio"),
    ("Servizio scadente. Mai piu", "Servizio scadente. Mai più"),
    ("Ottima qualità, prezzo onesto", "Ottima qualità, prezzo onesto"),
    ("Troppo caro per quello che offre!!!", "Troppo caro per quello che offre!"),
    ("un po' lento ma funziona", "un po' lento ma funziona"),
    ("Delusa dall'assistenza  #delusione", "Delusa dall'assistenza"),
]

ABREVIATURAS = {"es": {"q": "que", "xq": "porque", "pq": "porque", "tb": "también"},
                "it": {"nn": "non", "ke": "che", "cmq": "comunque", "xke": "perché"}}

PRUEBA_RETO = [
    ("es", "@tienda ¡¡LLEGÓÓÓ ya!!  q bueno, cuesta 4,99 € 😍 https://t.co/x9",
     ["llegó", "ya", "que", "bueno", "cuesta", "4,99"]),
    ("es", "Es _muy_ malo, xq no funciona   ....",
     ["es", "muy", "malo", "porque", "no", "funciona"]),
    ("es", "¿Dónde está el pedidooo del Sr. Ruiz? #urgente",
     ["dónde", "está", "el", "pedido", "del", "sr", "ruiz"]),
    ("es", "Pagué 1.250,50 euros y 3,5 % de comisión 😡😡",
     ["pagué", "1.250,50", "euros", "y", "3,5", "de", "comisión"]),
    ("it", "L'ordine nn è arrivato, cmq l'assistenza risponde!!! 👍",
     ["l'", "ordine", "non", "è", "arrivato", "comunque", "l'", "assistenza", "risponde"]),
    ("it", "BELLISSIMOOO, un'ora d'attesa 😍 www.sito.it/r/12",
     ["bellissimo", "un'", "ora", "d'", "attesa"]),
    ("it", "Il _Resegone_ costa 12.5 euro, ke bello   @negozio",
     ["il", "resegone", "costa", "12.5", "euro", "che", "bello"]),
    ("it", "dell'amico  Prezzo:   4,99!!!! Perfettoooo #consigliato",
     ["dell'", "amico", "prezzo", "4,99", "perfetto"]),
]

## Esercizio 1 — NFC, NFD e lunghezza

`PALABRA_NFC` (`canción`) è scritta con la `ó` come **un** punto di codice; `PALABRA_NFD` è la stessa parola con la `o` e un accento combinante. Calcola con `unicodedata`:

- `cadena_nfd`: `PALABRA_NFC` convertita in NFD,
- `lon_nfc`, `lon_nfd`: la lunghezza delle due versioni,
- `iguales_directo`: se `PALABRA_NFC == PALABRA_NFD` (senza normalizzare),
- `iguales_normalizando`: se sono uguali dopo aver normalizzato **entrambe** in NFC,
- `n_combinantes`: quanti caratteri di categoria `"Mn"` (segni combinanti) ha `PALABRA_NFD`.

Prima di programmare, prevedi a mano i cinque risultati.

In [ ]:
cadena_nfd = None            # TODO: PALABRA_NFC in forma NFD
lon_nfc = None               # TODO
lon_nfd = None               # TODO
iguales_directo = None       # TODO: confronto senza normalizzare
iguales_normalizando = None  # TODO: confronta dopo aver normalizzato le due in NFC
n_combinantes = None         # TODO: usa unicodedata.category(c) == "Mn"

# Controllo automatico
assert all(v is not None for v in (cadena_nfd, lon_nfc, lon_nfd, iguales_directo, iguales_normalizando, n_combinantes)), "Completa le righe contrassegnate con TODO"
assert cadena_nfd == PALABRA_NFD, "NFD scompone la vocale accentata in vocale + accento combinante"
assert (lon_nfc, lon_nfd) == (7, 8), "la versione NFD ha un punto di codice in più"
assert iguales_directo is False and iguales_normalizando is True, "diverse senza normalizzare, uguali normalizzando"
assert n_combinantes == 1, "c'è una sola marca combinante: l'accento"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Una funzione `limpiar` con espressioni regolari

Completa i quattro pattern (`PATRON_URL`, `PATRON_MENCION_HASHTAG`, `PATRON_ALARGADA`, `PATRON_ESPACIO_ANTES`) in modo che `limpiar_simple` faccia quanto segue:

1. togliere gli URL (`http://`, `https://` o `www.`) fino al successivo spazio,
2. togliere `@menzioni` e `#hashtag`,
3. ridurre a una le **lettere** ripetute tre o più volte (mai le cifre!: `1000` deve restare com'è); il pattern deve **catturare** la lettera in un gruppo,
4. togliere gli spazi che restano **prima** di `, ; : . ! ?` (catturando il segno in un gruppo).

Gli altri passi (ridurre `!!!`, `....` e spazi multipli) sono già scritti. `CASOS_LIMPIEZA` ha cinque coppie (ingresso, risultato atteso), calcolabili a mano.

In [ ]:
PATRON_URL = None                # TODO: http(s):// o www. seguito da non-spazi
PATRON_MENCION_HASHTAG = None    # TODO: @ o # seguito da caratteri di parola
PATRON_ALARGADA = None           # TODO: una lettera (non cifra né _) ripetuta 3 o più volte, catturata in un gruppo
PATRON_ESPACIO_ANTES = None      # TODO: spazi seguiti da uno tra , ; : . ! ? (catturato in un gruppo)

def limpiar_simple(texto):
    texto = re.sub(PATRON_URL, " ", texto)
    texto = re.sub(PATRON_MENCION_HASHTAG, " ", texto)
    texto = re.sub(PATRON_ALARGADA, r"\1", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    texto = re.sub(r"\.{4,}", "...", texto)
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(PATRON_ESPACIO_ANTES, r"\1", texto)
    return texto.strip()

# Controllo automatico
assert all(p is not None for p in (PATRON_URL, PATRON_MENCION_HASHTAG, PATRON_ALARGADA, PATRON_ESPACIO_ANTES)), "Completa le righe contrassegnate con TODO"
for entrada, esperado in CASOS_LIMPIEZA:
    assert limpiar_simple(entrada) == esperado, f"{entrada!r} → {limpiar_simple(entrada)!r} (atteso {esperado!r})"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Un tokenizzatore con `re.findall`

Scrivi `PATRON_TOKEN` in modo che `tokenizar` separi:

- i **numeri** con separatori decimali o delle migliaia come un unico pezzo (`4,99`, `12.5`, `1.250,50`),
- l'**elisione** dell'italiano: una parola seguita da apostrofo e da un'altra lettera resta come `l'`, `un'`, `dell'`, e poi viene la parola successiva,
- le **parole** (lettere e cifre `\w+`),
- qualunque altro carattere isolato (punteggiatura, `¿`, `¡`) come token a sé.

Prova l'ordine delle alternative: le più specifiche per prime. `CASOS_TOKEN` ha cinque frasi con il risultato atteso.

In [ ]:
PATRON_TOKEN = None      # TODO: alternative separate da |: numero | elisione | parola | altro simbolo

def tokenizar(texto):
    return re.findall(PATRON_TOKEN, texto)

# Controllo automatico
assert PATRON_TOKEN is not None, "Completa le righe contrassegnate con TODO"
for frase, esperado in CASOS_TOKEN:
    assert tokenizar(frase) == esperado, f"{frase!r} → {tokenizar(frase)}"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Token, tipi e hapax

Completa `resumir(tokens)` in modo che restituisca la quadrupla `(n_tokens, n_tipos, n_hapax, top)`, dove `top` è la coppia `(parola, frequenza)` del tipo più frequente. Si prova con due frasi brevi già tokenizzate (`TOKENS_ES` e `TOKENS_IT`), che puoi contare a mano.

In [ ]:
def resumir(tokens):
    conteo = Counter(tokens)
    n_tokens = None     # TODO: numero di occorrenze
    n_tipos = None      # TODO: numero di parole distinte
    n_hapax = None      # TODO: tipi che compaiono esattamente una volta
    top = None          # TODO: (parola, frequenza) del tipo più frequente
    return n_tokens, n_tipos, n_hapax, top

# Controllo automatico
assert resumir(TOKENS_ES)[0] is not None, "Completa le righe contrassegnate con TODO"
assert resumir(TOKENS_ES) == (12, 8, 5, ("el", 3)), f"rivedi il conteggio nella frase in spagnolo: {resumir(TOKENS_ES)}"   # dato-es
assert resumir(TOKENS_IT) == (11, 8, 6, ("il", 3)), f"rivedi il conteggio nella frase in italiano: {resumir(TOKENS_IT)}"
assert round(resumir(TOKENS_ES)[1] / resumir(TOKENS_ES)[0], 3) == 0.667, "TTR = tipi / token = 8 / 12"
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Valutare una pulizia rispetto al gold

`PARES` ha 20 coppie `(testo grezzo, testo pulito di riferimento)` in spagnolo e in italiano. Completa:

- `f1_tokens(pred, gold)`: F1 tra due liste di token come **multiinsiemi** (comuni = somma dei minimi delle frequenze; precisione = comuni / len(pred); *recall* = comuni / len(gold); 0 se non ci sono comuni),
- `evaluar(limpiador, pares)`: restituisce `(exacta, f1_medio)`, dove `exacta` è la proporzione di coppie il cui risultato, dopo `normalizar_comparacion`, coincide con il gold, e `f1_medio` la media dell'F1 delle parole (`re.findall(r"\w+", ...)` in minuscolo).

Si confrontano due pulitori: `sin_limpiar` e `limpiador_base` (toglie URL, menzioni, hashtag e riduce `!!`).

> **Nota:** con 20 coppie scelte a mano si esercita solo la logica della valutazione; non è una valutazione reale.

In [ ]:
def normalizar_comparacion(texto):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto).casefold()).strip()

def sin_limpiar(texto):
    return texto

def limpiador_base(texto):
    texto = re.sub(r"(?:https?://|www\.)\S+", " ", texto)
    texto = re.sub(r"[@#]\w+", " ", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def f1_tokens(pred, gold):
    comunes = None       # TODO: somma dei minimi delle frequenze (Counter & Counter)
    if not comunes:
        return 0.0
    precision = None     # TODO
    recall = None        # TODO
    return 2 * precision * recall / (precision + recall)

def evaluar(limpiador, pares):
    aciertos, f1s = [], []
    for bruto, gold in pares:
        pred = normalizar_comparacion(limpiador(bruto))
        gold_n = normalizar_comparacion(gold)
        aciertos.append(None)    # TODO: pred coincide con gold_n?
        f1s.append(f1_tokens(re.findall(r"\w+", pred), re.findall(r"\w+", gold_n)))
    return sum(aciertos) / len(pares), sum(f1s) / len(pares)

# Controllo automatico
assert f1_tokens(["a", "b", "c"], ["a", "b", "d"]) is not None, "Completa le righe contrassegnate con TODO"
assert abs(f1_tokens(["a", "b", "c"], ["a", "b", "d"]) - 2 / 3) < 1e-9, "2 comuni su 3 e su 3: F1 = 2/3"
assert abs(f1_tokens(["a", "a"], ["a"]) - 2 / 3) < 1e-9, "il multiinsieme conta 1 comune: precisione 1/2, recall 1"
assert f1_tokens([], ["a"]) == 0.0
exacta_sin, f1_sin = evaluar(sin_limpiar, PARES)
exacta_base, f1_base = evaluar(limpiador_base, PARES)
assert abs(exacta_sin - 0.4) < 1e-9 and abs(exacta_base - 0.6) < 1e-9, f"esatta: {exacta_sin}, {exacta_base}"
assert f1_base > f1_sin and f1_base > 0.9, f"F1: {f1_sin:.3f}, {f1_base:.3f}"
print(f"Esercizio 5 corretto ✔ (esatta {exacta_sin:.2f} → {exacta_base:.2f}; F1 {f1_sin:.3f} → {f1_base:.3f})")

---
## Sfida finale — `preparar(texto, idioma)`

Integra quanto visto: scrivi `preparar(texto, idioma)`, che riceve un testo grezzo e restituisce la **lista dei token di parola in minuscolo** (senza punteggiatura). Puoi riusare `limpiar_simple` e `tokenizar` degli esercizi 2 e 3, ma l'insieme di prova `PRUEBA_RETO` (otto testi, quattro per lingua, scritti per questa sfida) include cose che quegli esercizi non coprono: **emoji**, **marcatori `_corsivo_`** e **abbreviazioni da chat** (`ABREVIATURAS[idioma]` ti dà il dizionario della lingua).

Si valuta con l'**F1 medio dei token** e si richiede un minimo di 0,97.

> **Nota:** i passi e l'elenco di abbreviazioni sono stati scelti guardando queste stesse otto frasi: serve per esercitarsi, **non come valutazione reale**. Con testi nuovi l'F1 sarebbe inferiore; una valutazione onesta usa frasi che non hai guardato mentre progettavi la pulizia.

In [ ]:
def preparar(texto, idioma):
    # TODO 1: togli emoji e marcatori _corsivo_ (gli esercizi 2 e 3 non li coprono)
    # TODO 2: pulisci, passa al minuscolo e tokenizza
    # TODO 3: sostituisci le abbreviazioni con ABREVIATURAS[idioma] e lascia solo token di parola
    return None

# Controllo automatico
def f1_reto(pred, gold):
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if comunes == 0:
        return 0.0
    p, r = comunes / len(pred), comunes / len(gold)
    return 2 * p * r / (p + r)

f1s = []
for idioma, texto, esperado in PRUEBA_RETO:
    tokens = preparar(texto, idioma)
    assert tokens is not None, "Completa le righe contrassegnate con TODO"
    f1s.append(f1_reto(tokens, esperado))
f1_medio = sum(f1s) / len(f1s)
assert f1_medio >= 0.97, f"F1 medio {f1_medio:.3f}; casi peggiori: {[i for i, f in enumerate(f1s) if f < 0.9]}"
print(f"Sfida finale corretta ✔ (F1 medio {f1_medio:.3f} su {len(f1s)} testi)")

**Le tue conclusioni (scrivi 3-4 frasi):** quale caso della sfida ti è costato di più e perché? Cosa succederebbe al tuo `preparar` con un refuso (`esrellas`), con una parola accentata senza accento o con un'abbreviazione che non è nel dizionario? Quale informazione hai buttato pulendo (emoji, maiuscole, `!!!`) e in quale compito ti servirebbe?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
cadena_nfd = unicodedata.normalize("NFD", PALABRA_NFC)
lon_nfc = len(PALABRA_NFC)
lon_nfd = len(PALABRA_NFD)
iguales_directo = PALABRA_NFC == PALABRA_NFD
iguales_normalizando = unicodedata.normalize("NFC", PALABRA_NFC) == unicodedata.normalize("NFC", PALABRA_NFD)
n_combinantes = sum(unicodedata.category(c) == "Mn" for c in PALABRA_NFD)

# Controllo automatico
assert all(v is not None for v in (cadena_nfd, lon_nfc, lon_nfd, iguales_directo, iguales_normalizando, n_combinantes)), "Completa le righe contrassegnate con TODO"
assert cadena_nfd == PALABRA_NFD, "NFD scompone la vocale accentata in vocale + accento combinante"
assert (lon_nfc, lon_nfd) == (7, 8), "la versione NFD ha un punto di codice in più"
assert iguales_directo is False and iguales_normalizando is True, "diverse senza normalizzare, uguali normalizzando"
assert n_combinantes == 1, "c'è una sola marca combinante: l'accento"
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
PATRON_URL = r"(?:https?://|www\.)\S+"
PATRON_MENCION_HASHTAG = r"[@#]\w+"
PATRON_ALARGADA = r"([^\W\d_])\1{2,}"
PATRON_ESPACIO_ANTES = r"\s+([,;:.!?])"

def limpiar_simple(texto):
    texto = re.sub(PATRON_URL, " ", texto)
    texto = re.sub(PATRON_MENCION_HASHTAG, " ", texto)
    texto = re.sub(PATRON_ALARGADA, r"\1", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    texto = re.sub(r"\.{4,}", "...", texto)
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(PATRON_ESPACIO_ANTES, r"\1", texto)
    return texto.strip()

# Controllo automatico
assert all(p is not None for p in (PATRON_URL, PATRON_MENCION_HASHTAG, PATRON_ALARGADA, PATRON_ESPACIO_ANTES)), "Completa le righe contrassegnate con TODO"
for entrada, esperado in CASOS_LIMPIEZA:
    assert limpiar_simple(entrada) == esperado, f"{entrada!r} → {limpiar_simple(entrada)!r} (atteso {esperado!r})"
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
PATRON_TOKEN = r"\d+(?:[.,]\d+)*|\w+'(?=\w)|\w+|[^\w\s]"

def tokenizar(texto):
    return re.findall(PATRON_TOKEN, texto)

# Controllo automatico
assert PATRON_TOKEN is not None, "Completa le righe contrassegnate con TODO"
for frase, esperado in CASOS_TOKEN:
    assert tokenizar(frase) == esperado, f"{frase!r} → {tokenizar(frase)}"
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def resumir(tokens):
    conteo = Counter(tokens)
    n_tokens = len(tokens)
    n_tipos = len(conteo)
    n_hapax = sum(1 for v in conteo.values() if v == 1)
    top = conteo.most_common(1)[0]
    return n_tokens, n_tipos, n_hapax, top

# Controllo automatico
assert resumir(TOKENS_ES)[0] is not None, "Completa le righe contrassegnate con TODO"
assert resumir(TOKENS_ES) == (12, 8, 5, ("el", 3)), f"rivedi il conteggio nella frase in spagnolo: {resumir(TOKENS_ES)}"   # dato-es
assert resumir(TOKENS_IT) == (11, 8, 6, ("il", 3)), f"rivedi il conteggio nella frase in italiano: {resumir(TOKENS_IT)}"
assert round(resumir(TOKENS_ES)[1] / resumir(TOKENS_ES)[0], 3) == 0.667, "TTR = tipi / token = 8 / 12"
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
def normalizar_comparacion(texto):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto).casefold()).strip()

def sin_limpiar(texto):
    return texto

def limpiador_base(texto):
    texto = re.sub(r"(?:https?://|www\.)\S+", " ", texto)
    texto = re.sub(r"[@#]\w+", " ", texto)
    texto = re.sub(r"([!?])\1+", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def f1_tokens(pred, gold):
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if not comunes:
        return 0.0
    precision = comunes / len(pred)
    recall = comunes / len(gold)
    return 2 * precision * recall / (precision + recall)

def evaluar(limpiador, pares):
    aciertos, f1s = [], []
    for bruto, gold in pares:
        pred = normalizar_comparacion(limpiador(bruto))
        gold_n = normalizar_comparacion(gold)
        aciertos.append(pred == gold_n)
        f1s.append(f1_tokens(re.findall(r"\w+", pred), re.findall(r"\w+", gold_n)))
    return sum(aciertos) / len(pares), sum(f1s) / len(pares)

# Controllo automatico
assert f1_tokens(["a", "b", "c"], ["a", "b", "d"]) is not None, "Completa le righe contrassegnate con TODO"
assert abs(f1_tokens(["a", "b", "c"], ["a", "b", "d"]) - 2 / 3) < 1e-9, "2 comuni su 3 e su 3: F1 = 2/3"
assert abs(f1_tokens(["a", "a"], ["a"]) - 2 / 3) < 1e-9, "il multiinsieme conta 1 comune: precisione 1/2, recall 1"
assert f1_tokens([], ["a"]) == 0.0
exacta_sin, f1_sin = evaluar(sin_limpiar, PARES)
exacta_base, f1_base = evaluar(limpiador_base, PARES)
assert abs(exacta_sin - 0.4) < 1e-9 and abs(exacta_base - 0.6) < 1e-9, f"esatta: {exacta_sin}, {exacta_base}"
assert f1_base > f1_sin and f1_base > 0.9, f"F1: {f1_sin:.3f}, {f1_base:.3f}"
print(f"Esercizio 5 corretto ✔ (esatta {exacta_sin:.2f} → {exacta_base:.2f}; F1 {f1_sin:.3f} → {f1_base:.3f})")

### Soluzione della sfida finale

In [ ]:
def preparar(texto, idioma):
    texto = re.sub(r"[\U0001F300-\U0001FAFF☀-➿]+", " ", texto)
    texto = re.sub(r"(?<!\w)_(\S(?:.*?\S)?)_(?!\w)", r"\1", texto)
    tokens = tokenizar(limpiar_simple(texto).lower())
    tokens = [ABREVIATURAS[idioma].get(t, t) for t in tokens]
    return [t for t in tokens if re.match(r"\w", t)]

# Controllo automatico
def f1_reto(pred, gold):
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if comunes == 0:
        return 0.0
    p, r = comunes / len(pred), comunes / len(gold)
    return 2 * p * r / (p + r)

f1s = []
for idioma, texto, esperado in PRUEBA_RETO:
    tokens = preparar(texto, idioma)
    assert tokens is not None, "Completa le righe contrassegnate con TODO"
    f1s.append(f1_reto(tokens, esperado))
f1_medio = sum(f1s) / len(f1s)
assert f1_medio >= 0.97, f"F1 medio {f1_medio:.3f}; casi peggiori: {[i for i, f in enumerate(f1s) if f < 0.9]}"
print(f"Sfida finale corretta ✔ (F1 medio {f1_medio:.3f} su {len(f1s)} testi)")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (NFC/NFD, `casefold`, regex, token e tipi, valutazione di una pulizia)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*